# 10 - Cross-domain zero-shot ablation

Evaluates the system on 3,000 reviews from a balanced cross-domain corpus
(50% original reviews, 50% LLM-generated reviews) without adaptation. The
sample is the balanced `data/external_data.csv` file used in the paper, so it
is loaded as-is rather than resampled. Queries are encoded with
`sentence-transformers/all-distilroberta-v1` to match the original
experiments.

> **Reproduction note.** The released FAISS index was built with
> `roberta-base` while external queries use `all-distilroberta-v1`, so the
> similarity values are not directly comparable across encoders. The mismatch
> is preserved to reproduce the published numbers; see
> `src/spam_detection/embeddings.py`.

## Setup

In [ ]:
import pandas as pd

from spam_detection.classifier import load_roberta_model, predict_roberta_proba
from spam_detection.config import (
    CROSS_DOMAIN_PATH,
    K_NEIGHBORS,
    RESULTS_DIR,
    TAU_HIGH,
    TAU_LOW,
    VECTOR_DB_INDEX,
    VECTOR_DB_PARQUET,
)
from spam_detection.embeddings import encode_texts
from spam_detection.evaluation import comparison_table, gray_zone_analysis, predictions_from_results, save_table
from spam_detection.pipeline import run_ablation_study
from spam_detection.preprocessing import clean_text_basic, remove_empty_reviews
from spam_detection.prompts import create_hybrid_chain, create_light_chain
from spam_detection.retrieval import load_labeled_faiss_db, precompute_faiss_summaries
from spam_detection.utils import as_int_label, set_seed

set_seed(42)
RUN_LLM = True
OUTPUT_DIR = RESULTS_DIR / "runs" / "cross_domain_zero_shot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Preprocess the external corpus

In [ ]:
df = pd.read_csv(CROSS_DOMAIN_PATH)
df["text"] = df["text"].apply(clean_text_basic)
df = remove_empty_reviews(df, "text", min_length=3, verbose=False)
df["label"] = df["label"].map(as_int_label).astype(int)

print(df.shape)
print(df["label"].value_counts(normalize=True).round(4))

## Classifier probabilities and embeddings

In [ ]:
model, tokenizer = load_roberta_model()
clf_probas = predict_roberta_proba(df["text"].tolist(), model, tokenizer)
embeddings = encode_texts(df["text"].tolist(), batch_size=64, normalize=True)

faiss_db = load_labeled_faiss_db(VECTOR_DB_PARQUET, VECTOR_DB_INDEX)
faiss_summaries = precompute_faiss_summaries(embeddings, faiss_db, k_neighbors=K_NEIGHBORS)

## Run configurations

In [ ]:
y_true = df["label"].to_numpy()

clf_only = run_ablation_study(
    df=df, clf_probas=clf_probas, true_labels=y_true,
    use_llm=False, use_retrieval=False, use_interpolation=False, show_progress=False,
)
save_table(clf_only, OUTPUT_DIR / "clf_only.csv")

interpolation = run_ablation_study(
    df=df, clf_probas=clf_probas, true_labels=y_true, faiss_summaries=faiss_summaries,
    use_llm=False, use_retrieval=False, use_interpolation=True, show_progress=False,
)
save_table(interpolation, OUTPUT_DIR / "interpolation.csv")

if RUN_LLM:
    chain = create_hybrid_chain()
    full_rag = run_ablation_study(
        df=df, clf_probas=clf_probas, true_labels=y_true,
        llm_chain=chain, faiss_summaries=faiss_summaries,
        lower_thresh=TAU_LOW, upper_thresh=TAU_HIGH,
        use_llm=True, use_retrieval=True, use_interpolation=False,
    )
    save_table(full_rag, OUTPUT_DIR / "full_rag.csv")

    light_rag = run_ablation_study(
        df=df, clf_probas=clf_probas, true_labels=y_true,
        llm_chain=create_light_chain(), faiss_summaries=faiss_summaries,
        lower_thresh=TAU_LOW, upper_thresh=TAU_HIGH,
        use_llm=True, use_retrieval=True, use_interpolation=False,
    )
    save_table(light_rag, OUTPUT_DIR / "full_rag_light.csv")

## Results

In [ ]:
predictions = {
    "RoBERTa-LoRA": predictions_from_results(clf_only),
    "Interpolation (k-NN + LoRA)": predictions_from_results(interpolation),
}
if RUN_LLM:
    predictions["Full RAG"] = predictions_from_results(full_rag)
    predictions["Full RAG (light prompt)"] = predictions_from_results(light_rag)

table = comparison_table(y_true, predictions, dataset_name="Cross-Domain (zero-shot)")
display(table)
save_table(table, RESULTS_DIR / "tables" / "cross_domain_zero_shot_ablation.csv")

if RUN_LLM:
    print(gray_zone_analysis(full_rag, TAU_LOW, TAU_HIGH))